# Predicción de tasas brutas de mortalidad y natalidad con datos del Banco Mundial

**Proyecto interdisciplinario UIDE**  
**Integrantes:** Guillermo Paredes, Donato Oña, Mateo Villacreses  
**Docente:** Karla Estefania Moras Cajas

## Objetivo del cuaderno

Este notebook integra en un solo flujo el proyecto completo: obtención de datos vía API, comprensión del dataset, limpieza/preparación, análisis exploratorio, visualizaciones y comparación de modelos de aprendizaje automático.

## Relación con la rúbrica

| Criterio | Sección del notebook |
|---|---|
| Organización y uso adecuado de Markdown | Todo el cuaderno está dividido por secciones académicas |
| Obtención y comprensión de datos | Secciones 1 y 2 |
| Limpieza y preparación de datos | Sección 3 |
| Análisis exploratorio y visualizaciones | Sección 4 |
| Implementación y comparación de modelos | Sección 5 |


## 0. Configuración del entorno

En Google Colab se instalan las dependencias necesarias. En un entorno local o Codespaces, basta con haber instalado `requirements.txt`.


In [ ]:
import sys
import subprocess

if 'google.colab' in sys.modules:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'wbgapi', 'pyarrow', 'xgboost', 'lightgbm'
    ])


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wbgapi as wb

from IPython.display import display

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

try:
    import xgboost as xgb
except Exception as exc:
    xgb = None
    print(f"Aviso: XGBoost no disponible: {exc}")

try:
    import lightgbm as lgb
except Exception as exc:
    lgb = None
    print(f"Aviso: LightGBM no disponible: {exc}")

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Obtención de datos mediante API

Los datos provienen de **World Bank Open Data**, consultados mediante `wbgapi`. La unidad de análisis es el par **país-año** entre 2000 y 2022.

Las variables objetivo son:

- `death_rate`: tasa bruta de mortalidad por cada 1.000 habitantes.
- `birth_rate`: tasa bruta de natalidad por cada 1.000 habitantes.

Los predictores incluyen indicadores económicos, sociales, demográficos y contextuales.


In [ ]:
YEAR_START, YEAR_END = 2000, 2022

INDICATORS_BASE = {
    # Variables objetivo
    'SP.DYN.CDRT.IN':    'death_rate',
    'SP.DYN.CBRT.IN':    'birth_rate',

    # Predictores económicos
    'NY.GDP.PCAP.CD':    'gdp_per_capita',
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
    'SI.POV.GINI':       'gini',
    'SL.UEM.TOTL.ZS':    'unemployment',

    # Predictores sociales
    'SH.XPD.CHEX.GD.ZS': 'health_exp_pct_gdp',
    'SE.XPD.TOTL.GD.ZS': 'edu_exp_pct_gdp',
    'SP.URB.TOTL.IN.ZS': 'urban_pop_pct',
    'SP.POP.TOTL':       'population',
}

INDICATORS_AGE = {
    'SP.POP.65UP.TO.ZS': 'pop_65plus_pct',
    'SP.POP.0014.TO.ZS': 'pop_0to14_pct',
}

TARGETS = ['death_rate', 'birth_rate']

print(f"Periodo: {YEAR_START}-{YEAR_END}")
print(f"Indicadores base: {len(INDICATORS_BASE)}")
print(f"Indicadores de estructura etaria: {len(INDICATORS_AGE)}")


In [ ]:
def descargar_indicadores(indicadores, year_start=YEAR_START, year_end=YEAR_END):
    """Descarga indicadores del Banco Mundial y devuelve un DataFrame país-año."""
    raw = wb.data.DataFrame(
        series=list(indicadores.keys()),
        economy='all',
        time=range(year_start, year_end + 1),
        columns='series',
        skipBlanks=False,
        labels=False,
    ).reset_index()

    raw['year'] = raw['time'].str.replace('YR', '').astype(int)
    raw = raw.drop(columns='time').rename(columns={'economy': 'country_code', **indicadores})
    return raw[['country_code', 'year'] + list(indicadores.values())]

base = descargar_indicadores(INDICATORS_BASE)
age = descargar_indicadores(INDICATORS_AGE)

economies = pd.DataFrame([
    {
        'country_code': e['id'],
        'country_name': e['value'],
        'region': e['region'],
        'income_level': e['incomeLevel'],
    }
    for e in wb.economy.list()
])

region_clean = economies['region'].fillna('').astype(str).str.strip()
economies['is_country'] = region_clean.ne('') & region_clean.ne('NA')

# Se excluyen agregados regionales del Banco Mundial.
df = base.merge(economies, on='country_code', how='left')
df = df[df['is_country'].fillna(False)].drop(columns='is_country').copy()
df = df.merge(age, on=['country_code', 'year'], how='left')

meta_cols = ['country_code', 'country_name', 'region', 'income_level', 'year']
value_cols = list(INDICATORS_BASE.values()) + list(INDICATORS_AGE.values())
df = df[meta_cols + value_cols].reset_index(drop=True)

print(f"Shape inicial filtrado: {df.shape}")
print(f"Países únicos: {df['country_code'].nunique()}")
print(f"Años: {df['year'].min()}-{df['year'].max()}")
display(df.head())


## 2. Comprensión inicial del dataset

En esta sección se revisa el tamaño del dataset, el significado de las variables y la proporción de valores faltantes. Esto permite justificar la estrategia de limpieza posterior.


In [ ]:
diccionario = pd.DataFrame({
    'columna': meta_cols + value_cols,
    'rol': (
        ['metadata'] * len(meta_cols)
        + ['target', 'target']
        + ['predictor'] * (len(value_cols) - 2)
    ),
    'descripcion': [
        'Código ISO/economía del país',
        'Nombre del país',
        'Región geográfica del Banco Mundial',
        'Nivel de ingreso del Banco Mundial',
        'Año de observación',
        'Tasa bruta de mortalidad por cada 1.000 habitantes',
        'Tasa bruta de natalidad por cada 1.000 habitantes',
        'PIB per cápita en USD actuales',
        'Crecimiento anual del PIB (%)',
        'Coeficiente de Gini de ingresos/consumo',
        'Desempleo total (% fuerza laboral)',
        'Gasto corriente en salud (% PIB)',
        'Gasto público en educación (% PIB)',
        'Población urbana (% total)',
        'Población total',
        'Población de 65 años o más (% total)',
        'Población entre 0 y 14 años (% total)',
    ]
})

display(diccionario)


In [ ]:
print("Información general:")
display(pd.DataFrame({
    'filas': [df.shape[0]],
    'columnas': [df.shape[1]],
    'paises': [df['country_code'].nunique()],
    'anio_min': [df['year'].min()],
    'anio_max': [df['year'].max()],
}))

missing = pd.DataFrame({
    'n_missing': df[value_cols].isna().sum(),
    'pct_missing': (df[value_cols].isna().mean() * 100).round(2),
}).sort_values('pct_missing', ascending=False)

display(missing)


## 3. Limpieza y preparación de datos

La preparación sigue cinco pasos:

1. Eliminar filas sin variable objetivo.
2. Imputar predictores con una estrategia jerárquica que respeta la estructura país-año.
3. Crear una bandera de imputación para Gini.
4. Transformar variables sesgadas mediante logaritmo.
5. Codificar región, ingreso, año y pandemia para modelado.


In [ ]:
def imputacion_jerarquica(data, cols, flag_threshold=0.5):
    """Imputa valores faltantes usando continuidad por país y medianas por grupos."""
    data = data.copy().sort_values(['country_code', 'year']).reset_index(drop=True)
    flags_creados = []

    for col in cols:
        pct_missing = data[col].isna().mean()

        if pct_missing > flag_threshold:
            data[f'{col}_imputed'] = data[col].isna().astype(int)
            flags_creados.append(f'{col}_imputed')

        data[col] = data.groupby('country_code')[col].ffill()
        data[col] = data.groupby('country_code')[col].bfill()
        data[col] = data[col].fillna(data.groupby(['region', 'income_level'])[col].transform('median'))
        data[col] = data[col].fillna(data.groupby('region')[col].transform('median'))
        data[col] = data[col].fillna(data[col].median())

    return data, flags_creados

n_before = len(df)
df_model = df.dropna(subset=TARGETS).reset_index(drop=True)
print(f"Filas eliminadas por targets faltantes: {n_before - len(df_model)}")

predictor_cols = [c for c in value_cols if c not in TARGETS]
missing_antes = df_model[predictor_cols].isna().sum()
df_model, flags = imputacion_jerarquica(df_model, predictor_cols)
missing_despues = df_model[predictor_cols].isna().sum()

resumen_imp = pd.DataFrame({
    'faltantes_antes': missing_antes,
    'faltantes_despues': missing_despues,
    'pct_antes': (missing_antes / len(df_model) * 100).round(2),
    'pct_despues': (missing_despues / len(df_model) * 100).round(2),
}).sort_values('pct_antes', ascending=False)

display(resumen_imp)
print(f"Banderas de imputación creadas: {flags}")


In [ ]:
# Transformaciones logarítmicas para variables con cola larga.
SKEWED_COLS = ['gdp_per_capita', 'population']
for col in SKEWED_COLS:
    df_model[f'log_{col}'] = np.log1p(df_model[col])

df_model['year_norm'] = (df_model['year'] - YEAR_START) / (YEAR_END - YEAR_START)
df_model['is_pandemic'] = df_model['year'].isin([2020, 2021]).astype(int)

INCOME_ORDER = {'LIC': 0, 'LMC': 1, 'UMC': 2, 'HIC': 3}
df_model['income_level_ord'] = df_model['income_level'].map(INCOME_ORDER)

if df_model['income_level_ord'].isna().any():
    df_model['income_level_ord'] = df_model['income_level_ord'].fillna(
        df_model.groupby('region')['income_level_ord'].transform('median')
    )
    df_model['income_level_ord'] = df_model['income_level_ord'].fillna(df_model['income_level_ord'].median())

# One-hot de región.
region_ohe = pd.get_dummies(df_model['region'], prefix='region', dtype=int)
df_model = pd.concat([df_model, region_ohe], axis=1)
region_cols = region_ohe.columns.tolist()
assert 'region_' not in region_cols, "Se detectó una categoría de región vacía; revisar filtro de agregados."

FEATURES = (
    [c for c in predictor_cols if c not in SKEWED_COLS]
    + [f'log_{c}' for c in SKEWED_COLS]
    + ['year_norm', 'is_pandemic', 'income_level_ord']
    + region_cols
    + flags
)

assert df_model[FEATURES].isna().sum().sum() == 0, "Aún hay valores faltantes en features."

print(f"Dataset procesado: {df_model.shape}")
print(f"Features finales: {len(FEATURES)}")
for feat in FEATURES:
    print(f"- {feat}")


## 4. Análisis exploratorio y visualizaciones

Se exploran faltantes, distribución de objetivos, comportamiento regional y correlaciones. Estas visualizaciones ayudan a justificar la elección de modelos no lineales y la inclusión de estructura etaria.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
missing['pct_missing'].sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('% de valores faltantes')
ax.set_title('Valores faltantes por indicador en el dataset crudo')
ax.axvline(50, color='red', linestyle='--', alpha=0.6, label='50%')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for i, target in enumerate(TARGETS):
    sns.histplot(df_model[target], bins=40, kde=True, ax=axes[i, 0], color='steelblue')
    axes[i, 0].set_title(f'Distribución de {target}')
    axes[i, 0].set_xlabel(f'{target} por 1.000 habitantes')

    order = df_model.groupby('region')[target].median().sort_values().index
    sns.boxplot(data=df_model, y='region', x=target, order=order, ax=axes[i, 1], color='lightblue')
    axes[i, 1].set_title(f'{target} por región')
    axes[i, 1].set_xlabel(f'{target} por 1.000 habitantes')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, target in zip(axes, TARGETS):
    sns.lineplot(
        data=df_model,
        x='year',
        y=target,
        hue='region',
        estimator='median',
        errorbar=None,
        marker='o',
        markersize=4,
        ax=ax,
    )
    ax.set_title(f'Mediana anual de {target} por región')
    ax.set_ylabel(f'{target} por 1.000 habitantes')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
cols_corr = [c for c in FEATURES + TARGETS if not c.startswith('region_')]
corr = df_model[cols_corr].corr(method='spearman')

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlación de Spearman entre variables principales')
plt.tight_layout()
plt.show()

print("Top correlaciones absolutas con cada target:")
for target in TARGETS:
    top = corr[target].drop(TARGETS).abs().sort_values(ascending=False).head(8)
    print(f"\n{target}")
    display(top.to_frame('abs_spearman'))


## 5. Implementación y comparación de modelos

Se comparan cuatro enfoques de regresión:

- **Ridge:** línea base lineal regularizada.
- **Random Forest:** ensamble no lineal por bagging.
- **XGBoost:** boosting de árboles, fuerte en datos tabulares.
- **LightGBM:** boosting eficiente basado en histogramas.

La validación usa `GroupKFold` agrupado por país. Esto evita que años de un mismo país aparezcan simultáneamente en entrenamiento y validación.


In [ ]:
# Partición holdout por país.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df_model, groups=df_model['country_code']))

df_train = df_model.iloc[train_idx].reset_index(drop=True)
df_test = df_model.iloc[test_idx].reset_index(drop=True)

assert set(df_train['country_code']).isdisjoint(set(df_test['country_code']))

print(f"Train: {df_train.shape[0]} filas / {df_train['country_code'].nunique()} países")
print(f"Test:  {df_test.shape[0]} filas / {df_test['country_code'].nunique()} países")


In [ ]:
def construir_modelos(random_state=RANDOM_STATE):
    modelos = {
        'Ridge': Pipeline([
            ('scaler', StandardScaler()),
            ('reg', Ridge(alpha=1.0)),
        ]),
        'RandomForest': RandomForestRegressor(
            n_estimators=250,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1,
        ),
    }

    if xgb is not None:
        modelos['XGBoost'] = xgb.XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )

    if lgb is not None:
        modelos['LightGBM'] = lgb.LGBMRegressor(
            n_estimators=300,
            max_depth=-1,
            num_leaves=31,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )

    return modelos

X_train = df_train[FEATURES].values
groups_train = df_train['country_code'].values
cv = GroupKFold(n_splits=5)

scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2',
}

registros = []
for target in TARGETS:
    y_train = df_train[target].values
    for nombre, modelo in construir_modelos().items():
        scores = cross_validate(
            modelo,
            X_train,
            y_train,
            groups=groups_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
        )
        registros.append({
            'target': target,
            'modelo': nombre,
            'RMSE_mean': -scores['test_rmse'].mean(),
            'RMSE_std': scores['test_rmse'].std(),
            'MAE_mean': -scores['test_mae'].mean(),
            'R2_mean': scores['test_r2'].mean(),
            'R2_std': scores['test_r2'].std(),
        })

resultados = pd.DataFrame(registros).sort_values(['target', 'RMSE_mean']).reset_index(drop=True)
display(resultados.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, target in zip(axes, TARGETS):
    sub = resultados[resultados['target'] == target].sort_values('RMSE_mean')
    ax.barh(sub['modelo'], sub['RMSE_mean'], xerr=sub['RMSE_std'], color='steelblue', capsize=4)
    ax.invert_yaxis()
    ax.set_xlabel('RMSE promedio (GroupKFold)')
    ax.set_title(f'Comparación de modelos: {target}')
    for y_pos, (_, row) in enumerate(sub.iterrows()):
        ax.text(row['RMSE_mean'] * 1.01, y_pos, f"R²={row['R2_mean']:.2f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Predicciones fuera de pliegue para el mejor modelo de cada target.
mejores = resultados.loc[resultados.groupby('target')['RMSE_mean'].idxmin()].reset_index(drop=True)
display(mejores[['target', 'modelo', 'RMSE_mean', 'R2_mean']].round(3))

predicciones_oof = {}
for _, row in mejores.iterrows():
    target = row['target']
    modelo_nombre = row['modelo']
    modelo = construir_modelos()[modelo_nombre]
    y_train = df_train[target].values
    y_pred = cross_val_predict(modelo, X_train, y_train, groups=groups_train, cv=cv, n_jobs=-1)
    predicciones_oof[target] = (modelo_nombre, y_train, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (target, (modelo_nombre, y_true, y_pred)) in zip(axes, predicciones_oof.items()):
    ax.scatter(y_true, y_pred, alpha=0.35, s=14, color='steelblue')
    lim_min = min(y_true.min(), y_pred.min())
    lim_max = max(y_true.max(), y_pred.max())
    ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', lw=1)
    ax.set_xlabel(f'{target} real')
    ax.set_ylabel(f'{target} predicho')
    ax.set_title(
        f'{target} - {modelo_nombre}\n'
        f'RMSE={np.sqrt(mean_squared_error(y_true, y_pred)):.3f}, R²={r2_score(y_true, y_pred):.3f}'
    )

plt.tight_layout()
plt.show()


In [ ]:
# Ablación: desempeño con y sin estructura etaria.
FEATURES_SIN_EDAD = [c for c in FEATURES if c not in ['pop_0to14_pct', 'pop_65plus_pct']]
modelo_ab = construir_modelos().get('XGBoost', construir_modelos()['RandomForest'])

ablacion = []
for target in TARGETS:
    y_train = df_train[target].values
    for etiqueta, feats in [('con_edad', FEATURES), ('sin_edad', FEATURES_SIN_EDAD)]:
        scores = cross_validate(
            modelo_ab,
            df_train[feats].values,
            y_train,
            groups=groups_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
        )
        ablacion.append({
            'target': target,
            'features': etiqueta,
            'RMSE': -scores['test_rmse'].mean(),
            'R2': scores['test_r2'].mean(),
        })

df_ablacion = pd.DataFrame(ablacion)
display(df_ablacion.round(3))


In [ ]:
# Importancia por permutación en el holdout de países.
X_test = df_test[FEATURES].values
importancias = {}

for _, row in mejores.iterrows():
    target = row['target']
    modelo_nombre = row['modelo']
    modelo = construir_modelos()[modelo_nombre]

    modelo.fit(X_train, df_train[target].values)
    pi = permutation_importance(
        modelo,
        X_test,
        df_test[target].values,
        n_repeats=8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scoring='neg_root_mean_squared_error',
    )
    importancias[target] = pd.DataFrame({
        'feature': FEATURES,
        'importance_mean': pi.importances_mean,
        'importance_std': pi.importances_std,
    }).sort_values('importance_mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (target, imp_df) in zip(axes, importancias.items()):
    top = imp_df.head(10).iloc[::-1]
    ax.barh(top['feature'], top['importance_mean'], xerr=top['importance_std'], color='seagreen', capsize=3)
    ax.set_title(f'Top 10 importancia por permutación: {target}')
    ax.set_xlabel('Pérdida de RMSE al permutar')

plt.tight_layout()
plt.show()


## 6. Conclusiones

1. El problema se formuló como **regresión supervisada** porque las tasas de mortalidad y natalidad son variables continuas.
2. Se usó `GroupKFold` por país para evitar fuga de información en datos panel.
3. La estructura etaria es clave: `pop_65plus_pct` ayuda a explicar mortalidad y `pop_0to14_pct` está muy asociada a natalidad.
4. Los modelos no lineales basados en árboles suelen superar a la línea base lineal, especialmente en mortalidad, donde las relaciones no son estrictamente lineales.
5. Las importancias de variables deben interpretarse como **importancia predictiva**, no como causalidad.
6. El proyecto es reproducible porque los datos se obtienen por API y el pipeline completo queda contenido en este notebook.

### Checklist final de la rúbrica

- Organización y Markdown: secciones numeradas y explicación metodológica.
- Obtención y comprensión de datos: API del Banco Mundial, diccionario de variables y faltantes.
- Limpieza y preparación: filtrado de agregados, imputación jerárquica, logs, codificación y partición por país.
- EDA y visualizaciones: faltantes, distribuciones, tendencias regionales y correlaciones.
- Modelos: Ridge, Random Forest, XGBoost, LightGBM, métricas, gráficos, ablación e importancias.
